In [152]:
import plotly.graph_objects as go
import pandas as pd
from pathlib import Path
import numpy as np
import json

path_beta = Path.cwd().parent / "nn_learning_intensity" / "trainset_beta"
path_mu = Path.cwd().parent / "nn_learning_intensity" / "trainset_mu"
path_alpha_m = Path.cwd().parent / "nn_learning_intensity" / "trainset_alpha_m"

from plotly.subplots import make_subplots
import sys

fig = make_subplots(rows=2, cols=2)

with open(path_mu / "param_info.json", "r") as f:
    json_data_mu = json.load(f)

with open(path_beta / "param_info.json", "r") as f:
    json_data_beta = json.load(f)

with open(path_alpha_m / "param_info.json", "r") as f:
    json_data_alpha_m = json.load(f)

def freq_std(freqs, values):
    sf = np.sum(freqs) + 1e-7
    mean = np.dot(freqs, values) / sf
    var = np.dot((values - mean)**2, freqs) / sf
    return np.sqrt(var)

first_quote = pd.to_datetime("2023-03-01 16:00:00") # treat as starting date
last_quote = pd.to_datetime("2023-03-31 16:00:00") # treat as last date
ref_diff = (last_quote - first_quote).total_seconds()

vols = []
mus = []
alpha_ts = []
alpha_ms = []
vols_betas = []
vols_alpha_t = []
vols_alpha_m = []
betas = []
temp = []
temp2 = []

for dir in path_mu.iterdir():
    if dir.name == "param_info.json":
        continue

    if dir.name not in json_data_mu.keys():
        continue

    volumes = np.load(dir / "traded_volumes.npy", allow_pickle=True) # (10, 10, 2, 100)
    mu = json.loads(json_data_mu[dir.name]["params"])["mu_intensity"]

    mus.append(mu)
    vols.append(np.mean(volumes))

def autocorr(x, lag=1):
    x = np.asarray(x)
    return np.corrcoef(x[lag:], x[:-lag])[0, 1]

def fano_bins(timeseries, binsize=20):
    means = []
    vars = []
    for idx in range(0, len(timeseries), binsize):
        bin = timeseries[idx:min(len(timeseries),idx+binsize)]
        means.append(np.mean(bin))
        vars.append(np.var(bin))
    return [v/m for v,m in zip(vars, means)]
    

def burst_bins(timeseries, binsize=20):
    means = []
    stds = []
    for idx in range(0, len(timeseries), binsize):
        bin = timeseries[idx:min(len(timeseries),idx+binsize)]
        means.append(np.mean(bin))
        stds.append(np.std(bin))
    return [(s-m)/(s+m) for s,m in zip(stds, means)]

def bin_sum(x, binsize):
    x = np.asarray(x)
    n = len(x) // binsize
    x = x[:n * binsize]
    return x.reshape(n, binsize).sum(axis=1)

for dir in path_beta.iterdir():
    if dir.name == "param_info.json":
        continue

    if dir.name not in json_data_beta.keys():
        continue

    volumes = np.load(dir / "traded_volumes.npy", allow_pickle=True) # (10, 10, 2, 100)
    beta = json.loads(json_data_mu[dir.name]["params"])["beta"]

    betas.append(beta)
    m = np.sum(volumes, axis=(0,1,2))
    k = 1
    T = 50
    p = 0.10
    #vols_betas.append(autocorr(m[:int(p*len(m))], lag=1))
    xb = bin_sum(m, binsize=5)
    fano = np.var(xb) / (np.mean(xb) + 1e-12)
    ac1 = np.corrcoef(xb[1:], xb[:-1])[0,1]
    temp.append(fano)
    temp2.append(ac1)

for dir in path_alpha_m.iterdir():
    if dir.name == "param_info.json":
        continue

    if dir.name not in json_data_alpha_m.keys():
        continue

    volumes = np.load(dir / "traded_volumes.npy", allow_pickle=True) # (10, 10, 2, 100)
    alpha_m = json.loads(json_data_alpha_m[dir.name]["params"])["alpha_moneyness"]
    alpha_ms.append(alpha_m)
    strikes = json_data_alpha_m[dir.name]["strikes"]
    assetdata = np.load(dir / "assetdata.npy", allow_pickle=True) # shape (2, T)
    spot_prices = assetdata[0, :] # (T)

    moneynesses = [(strike - spot) / spot for strike, spot in zip(strikes, spot_prices)]
    print(moneynesses)
    thresh = np.quantile(moneynesses, 0.70) - np.quantile(moneynesses, 0.30)
    
    m = np.mean(volumes, axis=(0,2)) # (10,100)
    x = []
    for t in range(100):
        m_this = m[:,t]
        m_near_atm = np.sum(m_this[np.abs(np.array(moneynesses)) <= thresh])
        if np.sum(m[:,t]) == 0.0:
            x.append(0.0)
        else:
            x.append(m_near_atm / np.sum(m[:,t]))
    vols_alpha_m.append(np.mean(x))

betas = np.linspace(0.2, 0.9, 50) # beta
alpha_ms = np.linspace(0.005, 0.025, 50)

fig.add_trace(go.Scatter(x=mus, y=vols, mode="markers", name="mu"), col=1, row=1)
fig.add_trace(go.Scatter(x=betas, y=temp, mode="markers", name="beta"), col=2, row=2)
fig.add_trace(go.Scatter(x=betas, y=temp2, mode="markers", name="beta"), col=1, row=2)
fig.add_trace(go.Scatter(x=alpha_ms, y=vols_alpha_m, mode="markers", name="beta"), col=2, row=1)

# for val,val2 in zip(temp,temp2):
#     fig.add_trace(go.Scatter(x=list(range(len(val))), y=val, mode="markers", name="beta"), col=2, row=1)
#     fig.add_trace(go.Scatter(x=list(range(len(val2))), y=val2, mode="markers", name="beta"), col=2, row=1)

fig.show()

[np.float64(0.19230769230769232), np.float64(0.008566050484480019), np.float64(0.3478882987792725), np.float64(-0.0540960154175281), np.float64(-0.06075142625026507), np.float64(0.02438606634861557), np.float64(0.1923590441211315), np.float64(0.11869791366672669), np.float64(0.12238933094640363), np.float64(0.25623954719375275)]
[np.float64(0.19230769230769232), np.float64(-0.00183680587576407), np.float64(0.30339174561998455), np.float64(-0.08353880222721768), np.float64(-0.0803911697333014), np.float64(-0.001313769001699989), np.float64(0.16082819457450495), np.float64(0.07242573581382071), np.float64(0.0683751495293016), np.float64(0.22658960108620005)]
[np.float64(0.19230769230769232), np.float64(0.011063909957143058), np.float64(0.3115823216990964), np.float64(-0.07678531745572775), np.float64(-0.0673411290714923), np.float64(0.015287673318226768), np.float64(0.15914469056849923), np.float64(0.09538608382855915), np.float64(0.11342098755418123), np.float64(0.2669300226936775)]
[np

In [146]:
print(vols_alpha_m)

[np.float64(0.2408885711468074), np.float64(0.23532457456055275), np.float64(0.24559081525607354), np.float64(0.23123740644817925), np.float64(0.23966061002460814), np.float64(0.2342863919797964), np.float64(0.22951592615441238), np.float64(0.2177252410125044), np.float64(0.24047063763458565), np.float64(0.23239388801599908), np.float64(0.2275858581100418), np.float64(0.23941612372541993), np.float64(0.22909252969333507), np.float64(0.2512248669573453), np.float64(0.23086939403672083), np.float64(0.22279250453562052), np.float64(0.23425008581747883), np.float64(0.23854847957013203), np.float64(0.25131555043394305), np.float64(0.24458467585020913), np.float64(0.22340242174804298), np.float64(0.235675348118492), np.float64(0.20987692545802036), np.float64(0.23496881506761305), np.float64(0.2189165144779377), np.float64(0.2389273681016183), np.float64(0.2380769480124654), np.float64(0.24312306908484843), np.float64(0.24131759574721212), np.float64(0.2331758052314678), np.float64(0.2314540

In [132]:
print('temp:'); print(temp)
print('temp2:'); print(temp2)

temp:
[np.float64(2899.903721807856), np.float64(2238.205594331532), np.float64(2148.2313523917774), np.float64(2290.2561161492936), np.float64(2007.4332656123725), np.float64(2112.6516825778253), np.float64(2612.7475627614845), np.float64(2349.609296491665), np.float64(2980.451218753667), np.float64(2218.582536972038), np.float64(2346.552665872547), np.float64(3572.766423312439), np.float64(3297.0290954159614), np.float64(3040.4392959186207), np.float64(2916.4945213951555), np.float64(3764.9344546808093), np.float64(2834.3432893716645), np.float64(1877.329644972825), np.float64(2754.8790125205896), np.float64(2676.0468962792206), np.float64(2222.4239852398405), np.float64(2213.2672097716313), np.float64(2591.9593742324596), np.float64(2657.0010430527273), np.float64(2372.0790543215608), np.float64(2895.850700240107), np.float64(2737.64679256806), np.float64(3318.618467476825), np.float64(2458.377999294764), np.float64(3027.342442631905), np.float64(2370.3553065722917), np.float64(2576

In [138]:
from scipy.stats import pearsonr, spearmanr

r_fano, p_fano = spearmanr(betas, temp)
r_ac1, p_ac1 = spearmanr(betas, temp2)

print("beta vs fano:", r_fano, p_fano)
print("beta vs ac1:", r_ac1, p_ac1)

beta vs fano: 4.801920768307323e-05 0.9997359331556768
beta vs ac1: 0.06064825930372149 0.6756644275853207
